In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ML_CEP2 - Form responses 1.csv")

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns = [
    'timestamp',
    'location',
    'people_count',
    'month',
    'units_latest',
    'units_previous',
    'ac_hours',
    'fan_count',
    'fridge_age',
    'light_hours',
    'load_shedding_hours',
    'has_solar',
    'ac_count',
    'has_heater',
    'has_ups',
    'empty_column'
]

In [ ]:
df = df.drop('timestamp', axis=1)
df = df.drop('empty_column', axis=1)

In [ ]:
df.head()

In [ ]:
month_map = {
    'Jan': 'January', 'Feb': 'February', 'Mar': 'March',
    'Apr': 'April', 'May': 'May', 'Jun': 'June',
    'Jul': 'July', 'Aug': 'August', 'Sep': 'September',
    'Oct': 'October', 'Nov': 'November', 'Dec': 'December'
}
df['month'] = df['month'].apply(lambda x: month_map.get(x, x))

In [ ]:
location_map = {
    'Karachi - Urban': 'Main city',
    'Karachi - Suburban': 'Residential colony',
    'Main city area (Gulshan, DHA, Clifton, etc.)': 'Main city',
    'Residential colony or society': 'Residential colony',
    'Muhalla or local area': 'Muhalla or local area'
}
df['location'] = df['location'].map(location_map)

In [ ]:

categorical_cols = ['location', 'month', 'has_solar', 'has_heater', 'has_ups']
for col in categorical_cols:
    df[col] = df[col].astype(str)

In [ ]:
df['ac_count'] = df['ac_count'].fillna(df['ac_count'].median())
df['has_heater'] = df['has_heater'].fillna(df['has_heater'].mode()[0])
df['has_ups'] = df['has_ups'].fillna(df['has_ups'].mode()[0])
df['ac_count'] = df['ac_count'].astype(int)

In [ ]:
df.isna().sum()

In [ ]:
plt.style.use('default')
fig = plt.figure(figsize=(15, 10))
for i, col in enumerate(['units_latest', 'ac_hours', 'people_count', 'load_shedding_hours'], 1):
    plt.subplot(2, 2, i)
    plt.hist(df[col], bins=20, edgecolor='black', alpha=0.7)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(15, 10))
for i, col in enumerate(['location', 'month', 'has_solar', 'has_heater', 'has_ups'], 1):
    plt.subplot(2, 3, i)
    df[col].value_counts().plot(kind='bar')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# feature engineering
df['total_ac_load'] = df['ac_count'] * df['ac_hours']
df['units_diff'] = df['units_latest'] - df['units_previous']
df['consumption_change_pct'] = (df['units_diff'] / df['units_previous']) * 100

season_map = {
    'December': 'Winter', 'January': 'Winter', 'February': 'Winter',
    'March': 'Spring', 'April': 'Spring', 'May': 'Spring',
    'June': 'Summer', 'July': 'Summer', 'August': 'Summer',
    'September': 'Fall', 'October': 'Fall', 'November': 'Fall'
}
df['season'] = df['month'].map(season_map)

df['consumption_category'] = pd.cut(df['units_latest'],
                                     bins=[0, 200, 400, 1000],
                                     labels=['Low', 'Medium', 'High'])

In [ ]:
plt.figure(figsize=(8, 5))
avg_by_ac = df.groupby('ac_count')['units_latest'].mean()
plt.bar(avg_by_ac.index, avg_by_ac.values, color='skyblue', edgecolor='black')
plt.xlabel('Number of ACs')
plt.ylabel('Average Units Used')
plt.title('Average Energy Consumption by Number of ACs')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['ac_hours'], df['units_latest'], alpha=0.5, color='blue')
plt.xlabel('AC Hours per Day')
plt.ylabel('Units Used')
plt.title('AC Usage vs Energy Consumption')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
season_avg = df.groupby('season')['units_latest'].mean().reindex(['Summer', 'Winter', 'Spring', 'Fall'])
plt.bar(season_avg.index, season_avg.values, color='orange', edgecolor='black')
plt.xlabel('Season')
plt.ylabel('Average Units Used')
plt.title('Average Consumption by Season')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
df['pct_change'] = (df['units_diff'] / df['units_previous']) * 100

print("Percentage change statistics:")
print(df['pct_change'].describe())

def categorize_consumption(pct):
    if pct < 15:
        return 'Normal'
    elif 15 <= pct <= 40:
        return 'High'
    else:
        return 'Shock'

df['consumption_category_target'] = df['pct_change'].apply(categorize_consumption)

In [ ]:
df['solar'] = df['has_solar'].apply(lambda x: 1 if x == 'Yes' else 0)
df['heater'] = df['has_heater'].apply(lambda x: 1 if x == 'Yes' else 0)
df['ups'] = df['has_ups'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
print("DATA ANALYSIS SUMMARY")
print(f"\nTotal households analyzed: {len(df)}")
print(f"Average units consumed: {df['units_latest'].mean():.2f} units")
print(f"Range: {df['units_latest'].min():.2f} to {df['units_latest'].max():.2f} units")
print(f"\nAverage AC count: {df['ac_count'].mean():.2f}")
print(f"Households with solar panels: {(df['has_solar'] == 'Yes').sum()}")
print(f"\nFinal dataset columns:")
print(df.columns.tolist())

# Artificial Neural Networks (ANN)

Now after preprocessing we will implement 3 different custom neural network models for multiclass classification.

**Target Variable**: `consumption_category_target`

**Classes**: Normal, High, Shock

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD

import warnings
warnings.filterwarnings('ignore')

In [ ]:
model_df = df.copy()

model_df.head()

In [ ]:
# encode all non-numeric columns

encoder_dict = {}

cols_to_encode = model_df.select_dtypes(include=['object', 'category']).columns.tolist()

if 'consumption_category_target' in cols_to_encode:
    cols_to_encode.remove('consumption_category_target')

print("Columns being encoded:", cols_to_encode)

for col in cols_to_encode:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoder_dict[col] = le

model_df.head()

In [ ]:
# encode target variable to integers (0, 1, 2)

le_target = LabelEncoder()
model_df['consumption_category_target'] = le_target.fit_transform(model_df['consumption_category_target'])

print("Target classes:", le_target.classes_)
print("Encoded as    :", list(range(len(le_target.classes_))))
print()
print("Target distribution:")
print(model_df['consumption_category_target'].value_counts())

In [ ]:

Y = model_df['consumption_category_target']

# Drop target AND leaky columns
X = model_df.drop(['consumption_category_target', 'pct_change', 'units_diff', 'units_latest', 'consumption_change_pct'], axis=1)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print()
print("Leaky columns removed: pct_change, units_diff, consumption_change_pct")


## Class Imbalance Check and SMOTE

In [ ]:
# check class distribution

print("Class Distribution BEFORE SMOTE:")
counts_before = Y.value_counts().sort_index()
for i, count in counts_before.items():
    label = le_target.classes_[i]
    print(f"  {label}: {count} samples")

plt.figure(figsize=(6, 4))
plt.bar([le_target.classes_[i] for i in counts_before.index], counts_before.values,
        color=['steelblue', 'darkorange', 'green'], edgecolor='black')
plt.title('Class Distribution Before SMOTE')
plt.xlabel('Class')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# split data into train (70%), validation (15%), and test (15%)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, Y,
    test_size=0.30,
    random_state=42,
    stratify=Y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.50,
    random_state=42,
    stratify=y_train_raw
)

print(f"Before SMOTE - Training samples  : {X_train.shape[0]}")
print(f"Validation samples (unchanged)   : {X_val.shape[0]}")
print(f"Test samples (unchanged)         : {X_test.shape[0]}")


In [ ]:
# apply SMOTE only on training set

smote = SMOTE(random_state=42, k_neighbors=3)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(f"After SMOTE - Training samples: {X_train.shape[0]}")
print()
print("Class Distribution AFTER SMOTE (training only):")
counts_after = pd.Series(y_train).value_counts().sort_index()
for i, count in counts_after.items():
    label = le_target.classes_[i]
    print(f"  {label}: {count} samples")

plt.figure(figsize=(6, 4))
plt.bar([le_target.classes_[i] for i in counts_after.index], counts_after.values,
        color=['steelblue', 'darkorange', 'green'], edgecolor='black')
plt.title('Class Distribution After SMOTE (Training Set)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Scaling done")
print(f"Final training shape  : {X_train.shape}")
print(f"Final validation shape: {X_val.shape}")
print(f"Final test shape      : {X_test.shape}")


## Model 1: Simple ANN

Small network, very few neurons, no dropout.

In [ ]:
model1 = Sequential()

model1.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
model1.add(Dense(8, activation='relu'))

model1.add(Dense(3, activation='softmax'))

model1.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model1.summary()

In [ ]:
history1 = model1.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    verbose=1
)


In [ ]:
pred1 = model1.predict(X_test)
y_pred1 = np.argmax(pred1, axis=1)

acc1  = accuracy_score(y_test, y_pred1)
prec1 = precision_score(y_test, y_pred1, average='weighted', zero_division=0)
rec1  = recall_score(y_test, y_pred1, average='weighted', zero_division=0)
f11   = f1_score(y_test, y_pred1, average='weighted', zero_division=0)

print("Model 1 Results")
print()
print("Accuracy :", round(acc1, 4))
print("Precision:", round(prec1, 4))
print("Recall   :", round(rec1, 4))
print("F1 Score :", round(f11, 4))

## Model 2: Medium ANN

More neurons and a dropout layer.

In [ ]:
model2 = Sequential()

model2.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model2.add(Dropout(0.3))
model2.add(Dense(16, activation='relu'))

model2.add(Dense(3, activation='softmax'))

model2.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model2.summary()


In [ ]:
history2 = model2.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    verbose=1
)


In [ ]:
pred2 = model2.predict(X_test)
y_pred2 = np.argmax(pred2, axis=1)

acc2  = accuracy_score(y_test, y_pred2)
prec2 = precision_score(y_test, y_pred2, average='weighted', zero_division=0)
rec2  = recall_score(y_test, y_pred2, average='weighted', zero_division=0)
f12   = f1_score(y_test, y_pred2, average='weighted', zero_division=0)

print("Model 2 Results")
print()
print("Accuracy :", round(acc2, 4))
print("Precision:", round(prec2, 4))
print("Recall   :", round(rec2, 4))
print("F1 Score :", round(f12, 4))

## Model 3: Deep ANN

Deeper network with more layers and tuned dropout.

In [ ]:
model3 = Sequential()

model3.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model3.add(Dropout(0.2))
model3.add(Dense(32, activation='relu'))
model3.add(Dropout(0.2))
model3.add(Dense(16, activation='relu'))

model3.add(Dense(3, activation='softmax'))

model3.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model3.summary()


In [ ]:
history3 = model3.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    verbose=1
)


In [ ]:
pred3 = model3.predict(X_test)
y_pred3 = np.argmax(pred3, axis=1)

acc3  = accuracy_score(y_test, y_pred3)
prec3 = precision_score(y_test, y_pred3, average='weighted', zero_division=0)
rec3  = recall_score(y_test, y_pred3, average='weighted', zero_division=0)
f13   = f1_score(y_test, y_pred3, average='weighted', zero_division=0)

print("Model 3 Results")
print()
print("Accuracy :", round(acc3, 4))
print("Precision:", round(prec3, 4))
print("Recall   :", round(rec3, 4))
print("F1 Score :", round(f13, 4))

## Confusion Matrices

In [ ]:
model_names = ['Model 1 (Simple)', 'Model 2 (Medium)', 'Model 3 (Deep)']

class_labels = le_target.classes_

preds = [y_pred1, y_pred2, y_pred3]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (y_pred, name) in enumerate(zip(preds, model_names)):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels,
                ax=axes[i])
    axes[i].set_title(f'{name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:

for y_pred, name in zip(preds, model_names):
    print(f"  {name}")
    print(classification_report(y_test, y_pred, target_names=class_labels, zero_division=0))

## Model Comparison

In [ ]:
model_names = ['Model 1 (Simple)', 'Model 2 (Medium)', 'Model 3 (Deep)']

shock_idx = list(le_target.classes_).index('Shock')
high_idx  = list(le_target.classes_).index('High')

def per_class_metrics(y_true, y_pred, idx):
    f1  = f1_score(y_true, y_pred, average=None, zero_division=0)[idx]
    rec = recall_score(y_true, y_pred, average=None, zero_division=0)[idx]
    return round(f1, 4), round(rec, 4)

shock_f1_ann,  shock_rec_ann  = zip(*[per_class_metrics(y_test, yp, shock_idx) for yp in [y_pred1, y_pred2, y_pred3]])
high_f1_ann,   high_rec_ann   = zip(*[per_class_metrics(y_test, yp, high_idx)  for yp in [y_pred1, y_pred2, y_pred3]])

results_ann = pd.DataFrame({
    'Model'       : model_names,
    'Shock F1'    : list(shock_f1_ann),
    'Shock Recall': list(shock_rec_ann),
    'High F1'     : list(high_f1_ann),
    'High Recall' : list(high_rec_ann),
    'Overall Acc' : [round(acc1, 4), round(acc2, 4), round(acc3, 4)],
    'Overall F1'  : [round(f11, 4), round(f12, 4), round(f13, 4)]
})

results_ann


In [ ]:

x      = np.arange(3)
w      = 0.35
fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(x - w/2, results_ann['Shock F1'],     w, label='Shock F1',     color='tomato',      edgecolor='black')
bars2 = ax.bar(x + w/2, results_ann['Shock Recall'], w, label='Shock Recall', color='lightsalmon', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('Score')
ax.set_title('ANN – Shock F1 Score / Recall Comparison')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


## Final Conclusion

In [ ]:
# Rank by Shock F1 (primary), then High F1 (secondary)
best_ann = results_ann.sort_values(
    by=['Shock F1', 'High F1'], ascending=False
).reset_index(drop=True)

print("ANN Models ranked by Shock F1 Score (primary) then High F1 Score (secondary):")
print()
print(best_ann[['Model', 'Shock F1', 'Shock Recall', 'High F1', 'High Recall']].to_string(index=False))
print()
print("Best ANN Model  :", best_ann.iloc[0]['Model'])
print("Shock F1 Score  :", best_ann.iloc[0]['Shock F1'])
print("Shock Recall    :", best_ann.iloc[0]['Shock Recall'])
print("High F1 Score   :", best_ann.iloc[0]['High F1'])
print("High Recall     :", best_ann.iloc[0]['High Recall'])

best_ann_model_name = best_ann.iloc[0]['Model']


#Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:

Y_lr = model_df['consumption_category_target']
X_lr = model_df.drop([
    'consumption_category_target',
    'pct_change',
    'units_diff',
    'consumption_change_pct',
    'units_latest',
    'consumption_category'
], axis=1)

print(f"X_lr shape: {X_lr.shape}")
print(f"Y_lr shape: {Y_lr.shape}")

In [ ]:

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, Y_lr, test_size=0.30, random_state=42, stratify=Y_lr
)

X_train_lr, X_val_lr, y_train_lr, y_val_lr = train_test_split(
    X_train_lr, y_train_lr, test_size=0.50, random_state=42, stratify=y_train_lr
)

print(f"Training samples  : {X_train_lr.shape[0]}")
print(f"Validation samples: {X_val_lr.shape[0]}")
print(f"Test samples      : {X_test_lr.shape[0]}")

In [ ]:

scaler_lr = StandardScaler()

X_train_lr = scaler_lr.fit_transform(X_train_lr)
X_val_lr   = scaler_lr.transform(X_val_lr)
X_test_lr  = scaler_lr.transform(X_test_lr)

print("Scaling done")

## Model 1: Default Regularization (C=1.0)
Baseline model with default regularization strength and no class weighting bias.

In [ ]:

model_lr1 = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced', random_state=42)
model_lr1.fit(X_train_lr, y_train_lr)

pred_lr1 = model_lr1.predict(X_test_lr)

acc_lr1  = accuracy_score(y_test_lr, pred_lr1)
prec_lr1 = precision_score(y_test_lr, pred_lr1, average='weighted', zero_division=0)
rec_lr1  = recall_score(y_test_lr, pred_lr1, average='weighted', zero_division=0)
f1_lr1   = f1_score(y_test_lr, pred_lr1, average='weighted', zero_division=0)

print("Model 1 Results  (C=1.0)")
print()
print("Accuracy :", round(acc_lr1, 4))
print("Precision:", round(prec_lr1, 4))
print("Recall   :", round(rec_lr1, 4))
print("F1 Score :", round(f1_lr1, 4))

## Model 2: Strong Regularization (C=0.01)

In [ ]:

model_lr2 = LogisticRegression(C=0.01, max_iter=1000, class_weight='balanced', random_state=42)
model_lr2.fit(X_train_lr, y_train_lr)

pred_lr2 = model_lr2.predict(X_test_lr)

acc_lr2  = accuracy_score(y_test_lr, pred_lr2)
prec_lr2 = precision_score(y_test_lr, pred_lr2, average='weighted', zero_division=0)
rec_lr2  = recall_score(y_test_lr, pred_lr2, average='weighted', zero_division=0)
f1_lr2   = f1_score(y_test_lr, pred_lr2, average='weighted', zero_division=0)

print("Model 2 Results  (C=0.01)")
print()
print("Accuracy :", round(acc_lr2, 4))
print("Precision:", round(prec_lr2, 4))
print("Recall   :", round(rec_lr2, 4))
print("F1 Score :", round(f1_lr2, 4))

## Model 3: Weak Regularization (C=10.0)

In [ ]:

model_lr3 = LogisticRegression(C=10.0, max_iter=500, class_weight='balanced', random_state=42)
model_lr3.fit(X_train_lr, y_train_lr)

pred_lr3 = model_lr3.predict(X_test_lr)

acc_lr3  = accuracy_score(y_test_lr, pred_lr3)
prec_lr3 = precision_score(y_test_lr, pred_lr3, average='weighted', zero_division=0)
rec_lr3  = recall_score(y_test_lr, pred_lr3, average='weighted', zero_division=0)
f1_lr3   = f1_score(y_test_lr, pred_lr3, average='weighted', zero_division=0)

print("Model 3 Results  (C=10.0)")
print()
print("Accuracy :", round(acc_lr3, 4))
print("Precision:", round(prec_lr3, 4))
print("Recall   :", round(rec_lr3, 4))
print("F1 Score :", round(f1_lr3, 4))

## Confusion Matrices

In [ ]:

preds_lr    = [pred_lr1, pred_lr2, pred_lr3]
model_names_lr = ['Model 1 (C=1.0)', 'Model 2 (C=0.01)', 'Model 3 (C=10.0)']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (y_pred, name) in enumerate(zip(preds_lr, model_names_lr)):
    cm = confusion_matrix(y_test_lr, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels,
                ax=axes[i])
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices - Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

## Classification Reports

In [ ]:

for y_pred, name in zip(preds_lr, model_names_lr):
    print(f"  {name}")
    print(classification_report(y_test_lr, y_pred, target_names=class_labels, zero_division=0))

## Model Comparison

In [ ]:
model_names_lr = ['Model 1 (C=1.0)', 'Model 2 (C=0.01)', 'Model 3 (C=10.0)']

shock_f1_lr,  shock_rec_lr  = zip(*[per_class_metrics(y_test_lr, yp, shock_idx) for yp in [pred_lr1, pred_lr2, pred_lr3]])
high_f1_lr,   high_rec_lr   = zip(*[per_class_metrics(y_test_lr, yp, high_idx)  for yp in [pred_lr1, pred_lr2, pred_lr3]])

results_lr_comp = pd.DataFrame({
    'Model'          : model_names_lr,
    'Regularization' : ['Default (C=1.0)', 'Strong (C=0.01)', 'Weak (C=10.0)'],
    'Shock F1'       : list(shock_f1_lr),
    'Shock Recall'   : list(shock_rec_lr),
    'High F1'        : list(high_f1_lr),
    'High Recall'    : list(high_rec_lr),
    'Overall Acc'    : [round(acc_lr1, 4), round(acc_lr2, 4), round(acc_lr3, 4)],
    'Overall F1'     : [round(f1_lr1, 4), round(f1_lr2, 4), round(f1_lr3, 4)]
})

results_lr_comp


In [ ]:

x_lr = np.arange(3)
w_lr = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(x_lr - w_lr/2, results_lr_comp['Shock F1'],     w_lr, label='Shock F1',     color='tomato',      edgecolor='black')
bars2 = ax.bar(x_lr + w_lr/2, results_lr_comp['Shock Recall'], w_lr, label='Shock Recall', color='lightsalmon', edgecolor='black')

ax.set_xticks(x_lr)
ax.set_xticklabels(model_names_lr)
ax.set_ylabel('Score')
ax.set_title('Logistic Regression – Shock F1 Score / Recall Comparison')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


## Final Conclusion

In [ ]:
# Rank by Shock F1 (primary), then High F1 (secondary)
best_lr_comp = results_lr_comp.sort_values(
    by=['Shock F1', 'High F1'], ascending=False
).reset_index(drop=True)

print("Logistic Regression Models ranked by Shock F1 Score (primary) then High F1 Score (secondary):")
print()
print(best_lr_comp[['Model', 'Shock F1', 'Shock Recall', 'High F1', 'High Recall']].to_string(index=False))
print()
print("Best LR Model   :", best_lr_comp.iloc[0]['Model'])
print("Shock F1 Score  :", best_lr_comp.iloc[0]['Shock F1'])
print("Shock Recall    :", best_lr_comp.iloc[0]['Shock Recall'])
print("High F1 Score   :", best_lr_comp.iloc[0]['High F1'])
print("High Recall     :", best_lr_comp.iloc[0]['High Recall'])

best_lr_model_name = best_lr_comp.iloc[0]['Model']


## Decision Tree Classification

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
le_loc = LabelEncoder()
le_mon = LabelEncoder()
le_sea = LabelEncoder()

df['location_enc'] = le_loc.fit_transform(df['location'])
df['month_enc']    = le_mon.fit_transform(df['month'])
df['season_enc']   = le_sea.fit_transform(df['season'])

df['units_per_person'] = df['units_latest'] / df['people_count']

print("Class distribution:")
print(df['consumption_category_target'].value_counts())
print()
print("Baseline accuracy if model always predicts Normal:", round((df['consumption_category_target'] == 'Normal').mean(), 4))

##Remove the features which will cause data leakage


In [ ]:
features = [
    'people_count', 'units_previous',
    'ac_hours', 'fan_count', 'fridge_age', 'light_hours',
    'load_shedding_hours', 'ac_count', 'total_ac_load',
    'solar', 'heater', 'ups',
    'location_enc', 'month_enc', 'season_enc'
]

X_dt = df[features]
y_dt = df['consumption_category_target']

X_train_dt, X_temp_dt, y_train_dt, y_temp_dt = train_test_split(
    X_dt, y_dt, test_size=0.3, random_state=42, stratify=y_dt
)

X_val_dt, X_test_dt, y_val_dt, y_test_dt = train_test_split(
    X_temp_dt, y_temp_dt, test_size=0.5, random_state=42, stratify=y_temp_dt
)

print("Training samples:", len(X_train_dt))
print("Validation samples:", len(X_val_dt))
print("Testing samples :", len(X_test_dt))

##Model 1: max_depth=3, min_samples_split=2



In [ ]:
dt1 = DecisionTreeClassifier(max_depth=3, criterion='gini', min_samples_split=2,
                             class_weight='balanced', random_state=42)
dt1.fit(X_train_dt, y_train_dt)

train_pred1_dt = dt1.predict(X_train_dt)
val_pred1_dt   = dt1.predict(X_val_dt)
test_pred1_dt  = dt1.predict(X_test_dt)

print("Model 1 — Train Accuracy     :", round(accuracy_score(y_train_dt, train_pred1_dt), 4))
print("Model 1 — Validation Accuracy :", round(accuracy_score(y_val_dt, val_pred1_dt), 4))
print("Model 1 — Test Accuracy      :", round(accuracy_score(y_test_dt, test_pred1_dt), 4))
print()
print(classification_report(y_val_dt, val_pred1_dt))

##Plot the confusion matrix for model 1


In [ ]:
plt.figure(figsize=(6, 5))
cm1 = confusion_matrix(y_test_dt, test_pred1_dt, labels=['Normal', 'High', 'Shock'])
ConfusionMatrixDisplay(cm1, display_labels=['Normal', 'High', 'Shock']).plot(cmap='Blues', colorbar=False)
plt.title('Model 1 Confusion Matrix (depth=3)')
plt.show()

print("Tree depth:", dt1.get_depth())
print("Leaf nodes:", dt1.get_n_leaves())

## Model 2: max_depth=5, min_samples_split=5

The tree will not split a node unless it has at least 5 samples to prevent overfitting on small groups.

In [ ]:
dt2 = DecisionTreeClassifier(max_depth=5, criterion='gini', min_samples_split=5,
                             class_weight='balanced', random_state=42)
dt2.fit(X_train_dt, y_train_dt)

train_pred2_dt = dt2.predict(X_train_dt)
val_pred2_dt   = dt2.predict(X_val_dt)
test_pred2_dt  = dt2.predict(X_test_dt)

print("Model 2 — Train Accuracy     :", round(accuracy_score(y_train_dt, train_pred2_dt), 4))
print("Model 2 — Validation Accuracy :", round(accuracy_score(y_val_dt, val_pred2_dt), 4))
print("Model 2 — Test Accuracy      :", round(accuracy_score(y_test_dt, test_pred2_dt), 4))
print()
print(classification_report(y_val_dt, val_pred2_dt))

##Plot confusion matrix for model 2

In [ ]:
plt.figure(figsize=(6, 5))
cm2 = confusion_matrix(y_test_dt, test_pred2_dt, labels=['Normal', 'High', 'Shock'])
ConfusionMatrixDisplay(cm2, display_labels=['Normal', 'High', 'Shock']).plot(cmap='Oranges', colorbar=False)
plt.title('Model 2 Confusion Matrix (depth=5)')
plt.show()

print("Tree depth:", dt2.get_depth())
print("Leaf nodes:", dt2.get_n_leaves())

## Model 3: max_depth=None, min_samples_split=10

Fully grown tree with no depth limit.

In [ ]:
dt3 = DecisionTreeClassifier(max_depth=None, criterion='gini', min_samples_split=10,
                             class_weight='balanced', random_state=42)
dt3.fit(X_train_dt, y_train_dt)

train_pred3_dt = dt3.predict(X_train_dt)
val_pred3_dt   = dt3.predict(X_val_dt)
test_pred3_dt  = dt3.predict(X_test_dt)

print("Model 3 — Train Accuracy     :", round(accuracy_score(y_train_dt, train_pred3_dt), 4))
print("Model 3 — Validation Accuracy :", round(accuracy_score(y_val_dt, val_pred3_dt), 4))
print("Model 3 — Test Accuracy      :", round(accuracy_score(y_test_dt, test_pred3_dt), 4))
print()
print(classification_report(y_val_dt, val_pred3_dt))

## Plot confusion matrix for model 3


In [ ]:
plt.figure(figsize=(6, 5))
cm3 = confusion_matrix(y_test_dt, test_pred3_dt, labels=['Normal', 'High', 'Shock'])
ConfusionMatrixDisplay(cm3, display_labels=['Normal', 'High', 'Shock']).plot(cmap='Greens', colorbar=False)
plt.title('Model 3 Confusion Matrix (full depth)')
plt.show()

print("Tree depth:", dt3.get_depth())
print("Leaf nodes:", dt3.get_n_leaves())

### Comparison of All Three Models

In [ ]:
results = pd.DataFrame({
    'Model': ['Model 1', 'Model 2', 'Model 3'],
    'max_depth': [3, 5, 'None'],
    'min_samples_split': [2, 5, 10],
    'Train Accuracy': [round(accuracy_score(y_train_dt, train_pred1_dt), 4),
                       round(accuracy_score(y_train_dt, train_pred2_dt), 4),
                       round(accuracy_score(y_train_dt, train_pred3_dt), 4)],
    'Validation Accuracy': [round(accuracy_score(y_val_dt, val_pred1_dt), 4),
                            round(accuracy_score(y_val_dt, val_pred2_dt), 4),
                            round(accuracy_score(y_val_dt, val_pred3_dt), 4)],
    'Test Accuracy': [round(accuracy_score(y_test_dt, test_pred1_dt), 4),
                      round(accuracy_score(y_test_dt, test_pred2_dt), 4),
                      round(accuracy_score(y_test_dt, test_pred3_dt), 4)],
})

print(results.to_string(index=False))
print()

In [ ]:
x = np.arange(3)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, results['Train Accuracy'], width, label='Train', color='steelblue')
ax.bar(x, results['Validation Accuracy'], width, label='Validation', color='orange')
ax.bar(x + width, results['Test Accuracy'], width, label='Test', color='green')

ax.set_xticks(x)
ax.set_xticklabels(results['Model'])
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.show()

##Plot F1_Score for high and shock

In [ ]:
high_f1_dt = [f1_score(y_val_dt, val_pred1_dt, labels=['High'], average=None)[0],
              f1_score(y_val_dt, val_pred2_dt, labels=['High'], average=None)[0],
              f1_score(y_val_dt, val_pred3_dt, labels=['High'], average=None)[0]]

shock_f1_dt = [f1_score(y_val_dt, val_pred1_dt, labels=['Shock'], average=None)[0],
               f1_score(y_val_dt, val_pred2_dt, labels=['Shock'], average=None)[0],
               f1_score(y_val_dt, val_pred3_dt, labels=['Shock'], average=None)[0]]

x = np.arange(3)
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, high_f1_dt, w, label='High F1', color='darkorange', edgecolor='black')
ax.bar(x + w/2, shock_f1_dt, w, label='Shock F1', color='tomato', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['Model 1 (depth=3)', 'Model 2 (depth=5)', 'Model 3 (full)'])
ax.set_ylabel('F1 Score')
ax.set_title('High and Shock F1 Score Comparison (Validation Set)')
ax.legend()
ax.set_ylim(0, 0.8)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

##Analysis

Model 1 (depth=3, min_samples_split=2)
The model is too simple for this problem. With only 3 levels of splits the tree could not separate three classes across 15 features. training accuracy itself was only 0.70 which means the model is not fitting the training data well.

Model 2 (depth=5, min_samples_split=5)
We initially used depth=7 but the train-validation gap was 0.15 so we reduced it to 5. The gap came down to 0.08 with the same validation accuracy, meaning better generalisation. High F1 is 0.29 and Shock F1 is 0.53 on the validation set, both the best across all three models.

Model 3 (depth=None, min_samples_split=10)
Highest validation accuracy at 0.79 but High F1 dropped to 0.19, meaning it is almost completely missing the High class. Shock samples causing unreliable predictions, but even with that fix the minority class performance is weaker than Model 2.

##Best Model of DT
We selected "Model 2" because it has the best F1 score for both High "0.29" and Shock "0.53" on the validation set. The purpose of this system is to detect abnormal bills so performance on these two classes matters the most than overall accuracy.

## Out of all 9, Best Model

In [ ]:
comparison_df = pd.DataFrame({
    'Algorithm': [
        'ANN (Deep - Model 3)',
        'Logistic Regression (C=0.01)',
        'Decision Tree (depth=5)'
    ],
    'Shock F1 Score'  : [0.5833, 0.6842, 0.5300],
    'Shock Recall'    : [0.4667, 0.8667, 0.3900],
    'High F1 Score'   : [0.2162, 0.2222, 0.2900],
    'High Recall'     : [0.2105, 0.3684, 0.2900],
    'Overall Accuracy': [0.8318, 0.7318, 0.7900]
})

comparison_df_sorted = comparison_df.sort_values(
    by=['Shock F1 Score', 'Shock Recall'], ascending=False
).reset_index(drop=True)

print("MODEL COMPARISON - BEST FROM EACH ALGORITHM")
print()
print(comparison_df_sorted.to_string(index=False))
print()

best_model = comparison_df_sorted.iloc[0]

print(f"BEST MODEL: {best_model['Algorithm']}")
print(f"\n   Shock F1 Score  : {best_model['Shock F1 Score']:.4f}")
print(f"   Shock Recall    : {best_model['Shock Recall']:.4f}")
print(f"   High F1 Score   : {best_model['High F1 Score']:.4f}")
print(f"   High Recall     : {best_model['High Recall']:.4f}")
print(f"   Overall Accuracy: {best_model['Overall Accuracy']:.4f}")

In [ ]:
algorithms   = comparison_df_sorted['Algorithm'].tolist()
shock_f1     = comparison_df_sorted['Shock F1 Score'].tolist()
shock_recall = comparison_df_sorted['Shock Recall'].tolist()

x_cmp = np.arange(len(algorithms))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x_cmp - width/2, shock_f1,     width, label='Shock F1 Score', color='tomato',      edgecolor='black')
bars2 = ax.bar(x_cmp + width/2, shock_recall, width, label='Shock Recall',   color='lightsalmon', edgecolor='black')

ax.set_xticks(x_cmp)
ax.set_xticklabels(algorithms, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Shock Class Performance Comparison')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

for i, (f1_val, rec_val) in enumerate(zip(shock_f1, shock_recall)):
    ax.annotate(f'{f1_val:.3f}', xy=(i - width/2, f1_val), xytext=(0, 5),
                textcoords='offset points', ha='center', va='bottom', fontsize=9)
    ax.annotate(f'{rec_val:.3f}', xy=(i + width/2, rec_val), xytext=(0, 5),
                textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
ranking = comparison_df_sorted.copy()
ranking['Rank'] = range(1, len(ranking) + 1)

print("FINAL RECOMMENDATION")
print()
print("Ranking (by Shock F1 Score then Shock Recall):")
print(ranking[['Rank', 'Algorithm', 'Shock F1 Score', 'Shock Recall']].to_string(index=False))
print()
print("LOGISTIC REGRESSION (C=0.01) is the BEST model because:")
print("   Highest Shock F1 Score (0.6842), best at detecting abnormal bills")
print("   Highest Shock Recall (0.8667), catches most shock events")


## Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("   ELECTRICITY CONSUMPTION PREDICTOR (LOGISTIC REGRESSION)")

location_w   = widgets.Dropdown(
    options=['Main city', 'Residential colony', 'Muhalla or local area'],
    description='Location:')

month_w      = widgets.Dropdown(
    options=['January','February','March','April','May','June',
             'July','August','September','October','November','December'],
    description='Month:')

has_solar_w  = widgets.Dropdown(options=['No', 'Yes'], description='Solar Panel:')
has_heater_w = widgets.Dropdown(options=['No', 'Yes'], description='Heater:')
has_ups_w    = widgets.Dropdown(options=['No', 'Yes'], description='UPS:')

people_count_w   = widgets.IntSlider(value=4,  min=1,  max=20,  description='People:')
ac_count_w       = widgets.IntSlider(value=1,  min=0,  max=10,  description='AC Count:')
fan_count_w      = widgets.IntSlider(value=3,  min=0,  max=20,  description='Fans:')

units_latest_w   = widgets.FloatText(value=300.0, description='Units (latest):')
units_previous_w = widgets.FloatText(value=250.0, description='Units (prev):')
ac_hours_w       = widgets.FloatText(value=6.0,   description='AC hrs/day:')
fridge_age_w     = widgets.FloatText(value=3.0,   description='Fridge age (yrs):')
light_hours_w    = widgets.FloatText(value=6.0,   description='Light hrs/day:')
load_shedding_w  = widgets.FloatText(value=2.0,   description='Load shed hrs:')

predict_btn = widgets.Button(description='Predict', button_style='success')
output_lr   = widgets.Output()

display(
    widgets.VBox([
        widgets.HTML("<b>Location & Time</b>"),
        location_w, month_w,
        widgets.HTML("<b>Household</b>"),
        people_count_w, ac_count_w, fan_count_w,
        widgets.HTML("<b>Consumption</b>"),
        units_latest_w, units_previous_w,
        widgets.HTML("<b>Appliances</b>"),
        ac_hours_w, fridge_age_w, light_hours_w, load_shedding_w,
        widgets.HTML("<b>Home Features</b>"),
        has_solar_w, has_heater_w, has_ups_w,
        predict_btn,
        output_lr
    ])
)

def on_predict_lr(b):
    with output_lr:
        clear_output()

        print("PREDICTION RESULTS (LOGISTIC REGRESSION)")

        season_map = {
            'December': 'Winter', 'January': 'Winter', 'February': 'Winter',
            'March': 'Spring',  'April': 'Spring',   'May': 'Spring',
            'June': 'Summer',   'July': 'Summer',    'August': 'Summer',
            'September': 'Fall','October': 'Fall',   'November': 'Fall'
        }

        season         = season_map[month_w.value]
        total_ac_load  = ac_count_w.value * ac_hours_w.value
        units_latest   = units_latest_w.value
        units_previous = units_previous_w.value

        pct_change = ((units_latest - units_previous) / units_previous * 100) if units_previous > 0 else 0

        if pct_change < 15:
            bill_label = 'Normal'
        elif pct_change <= 40:
            bill_label = 'High'
        else:
            bill_label = 'Shock'

        input_dict = {
            'location'            : location_w.value,
            'people_count'        : people_count_w.value,
            'month'               : month_w.value,
            'units_previous'      : units_previous,
            'ac_hours'            : ac_hours_w.value,
            'fan_count'           : fan_count_w.value,
            'fridge_age'          : fridge_age_w.value,
            'light_hours'         : light_hours_w.value,
            'load_shedding_hours' : load_shedding_w.value,
            'has_solar'           : has_solar_w.value,
            'ac_count'            : ac_count_w.value,
            'has_heater'          : has_heater_w.value,
            'has_ups'             : has_ups_w.value,
            'total_ac_load'       : total_ac_load,
            'solar'               : 1 if has_solar_w.value == 'Yes' else 0,
            'heater'              : 1 if has_heater_w.value == 'Yes' else 0,
            'ups'                 : 1 if has_ups_w.value == 'Yes' else 0,
            'season'              : season,
        }

        input_df = pd.DataFrame([input_dict])

        for col, le in encoder_dict.items():
            if col in input_df.columns:
                try:
                    input_df[col] = le.transform(input_df[col].astype(str))
                except ValueError:
                    input_df[col] = 0

        try:
            input_df['season'] = le_sea.transform(input_df['season'].astype(str))
        except ValueError:
            input_df['season'] = 0

        feature_cols = list(X_lr.columns)
        input_df     = input_df[feature_cols]
        input_scaled = scaler_lr.transform(input_df)

        probs      = model_lr2.predict_proba(input_scaled)[0]
        classes    = list(le_target.classes_)
        pred_label = bill_label
        confidence = probs[classes.index(pred_label)] * 100

        normal_prob = probs[classes.index('Normal')] * 100
        high_prob   = probs[classes.index('High')]   * 100
        shock_prob  = probs[classes.index('Shock')]  * 100

        print(f"\nInput Summary:")
        print(f"Location: {location_w.value}")
        print(f"Month: {month_w.value} ({season})")
        print(f"People: {people_count_w.value}")
        print(f"AC Count: {ac_count_w.value} (running {ac_hours_w.value} hrs/day)")
        print(f"Previous Units: {units_previous:.0f}")
        print(f"Latest Units:   {units_latest:.0f}")
        print(f"Change:         {pct_change:+.1f}%")
        print()
        print(f"PREDICTION: {pred_label}")
        print(f"Confidence: {confidence:.1f}%")
        print()
        print(f"Class Probabilities (model):")
        print(f"Normal: {normal_prob:.1f}%")
        print(f"High:   {high_prob:.1f}%")
        print(f"Shock:  {shock_prob:.1f}%")
        print()

        if pred_label == 'Shock':
            print("RECOMMENDATION: Unusual consumption spike detected!")
            print("Check for appliance malfunction or meter issues")
            print("Verify if new appliances were added")
            print("Consider energy audit")
        elif pred_label == 'High':
            print("RECOMMENDATION: High consumption detected")
            print("Reduce AC usage or increase temperature setting")
            print("Check for inefficient appliances")
            print("Consider energy-saving measures")
        else:
            print("RECOMMENDATION: Normal consumption pattern")
            print("Continue current usage habits")
            print("Monitor periodically for changes")

        print()
        print("Model: Logistic Regression (C=0.01) - Best for Shock Detection")

predict_btn.on_click(on_predict_lr)

## Best Model Split Testing

Testing Logistic Regression (C=0.01)

In [ ]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

Y_split = model_df['consumption_category_target']
X_split = model_df.drop([
    'consumption_category_target',
    'pct_change',
    'units_diff',
    'consumption_change_pct',
    'units_latest',
    'consumption_category'
], axis=1)

X_tr_8010, X_tmp_8010, y_tr_8010, y_tmp_8010 = train_test_split(
    X_split, Y_split, test_size=0.20, random_state=42, stratify=Y_split
)
X_val_8010, X_test_8010, y_val_8010, y_test_8010 = train_test_split(
    X_tmp_8010, y_tmp_8010, test_size=0.50, random_state=42, stratify=y_tmp_8010
)

smote_8010 = SMOTE(random_state=42, k_neighbors=3)
X_tr_8010_sm, y_tr_8010_sm = smote_8010.fit_resample(X_tr_8010, y_tr_8010)

scaler_8010 = StandardScaler()
X_tr_8010_sc   = scaler_8010.fit_transform(X_tr_8010_sm)
X_val_8010_sc  = scaler_8010.transform(X_val_8010)
X_test_8010_sc = scaler_8010.transform(X_test_8010)

print(f"80/10/10 Split:  Train: {X_tr_8010.shape[0]}  Val: {X_val_8010.shape[0]}  Test: {X_test_8010.shape[0]}")
print(f"After SMOTE:  Train: {X_tr_8010_sm.shape[0]}")


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_8010 = LogisticRegression(C=0.01, max_iter=1000, class_weight='balanced', random_state=42)
lr_8010.fit(X_tr_8010_sc, y_tr_8010_sm)

pred_8010 = lr_8010.predict(X_test_8010_sc)

print("LOGISTIC REGRESSION (C=0.01) 80/10/10 Split Results")
print()
print(f"Accuracy : {accuracy_score(y_test_8010, pred_8010):.4f}")
print(f"Precision: {precision_score(y_test_8010, pred_8010, average='weighted', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test_8010, pred_8010, average='weighted', zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_test_8010, pred_8010, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(y_test_8010, pred_8010, target_names=le_target.classes_, zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cm_8010 = confusion_matrix(y_test_8010, pred_8010)
sns.heatmap(cm_8010, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix LR (C=0.01) 80/10/10 Split')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
X_tr_6020, X_tmp_6020, y_tr_6020, y_tmp_6020 = train_test_split(
    X_split, Y_split, test_size=0.40, random_state=42, stratify=Y_split
)
X_val_6020, X_test_6020, y_val_6020, y_test_6020 = train_test_split(
    X_tmp_6020, y_tmp_6020, test_size=0.50, random_state=42, stratify=y_tmp_6020
)

smote_6020 = SMOTE(random_state=42, k_neighbors=3)
X_tr_6020_sm, y_tr_6020_sm = smote_6020.fit_resample(X_tr_6020, y_tr_6020)

scaler_6020 = StandardScaler()
X_tr_6020_sc   = scaler_6020.fit_transform(X_tr_6020_sm)
X_val_6020_sc  = scaler_6020.transform(X_val_6020)
X_test_6020_sc = scaler_6020.transform(X_test_6020)

print(f"60/20/20 Split  Train: {X_tr_6020.shape[0]}  Val: {X_val_6020.shape[0]}  Test: {X_test_6020.shape[0]}")
print(f"After SMOTE  Train: {X_tr_6020_sm.shape[0]}")


In [ ]:
lr_6020 = LogisticRegression(C=0.01, max_iter=1000, class_weight='balanced', random_state=42)
lr_6020.fit(X_tr_6020_sc, y_tr_6020_sm)

pred_6020 = lr_6020.predict(X_test_6020_sc)

print("LOGISTIC REGRESSION (C=0.01) 60/20/20 Split Results")
print()
print(f"Accuracy : {accuracy_score(y_test_6020, pred_6020):.4f}")
print(f"Precision: {precision_score(y_test_6020, pred_6020, average='weighted', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test_6020, pred_6020, average='weighted', zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_test_6020, pred_6020, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(y_test_6020, pred_6020, target_names=le_target.classes_, zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cm_6020 = confusion_matrix(y_test_6020, pred_6020)
sns.heatmap(cm_6020, annot=True, fmt='d', cmap='Purples',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix — LR (C=0.01) | 60/20/20 Split')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
split_summary_df = pd.DataFrame({
    'Split': ['70/15/15', '80/10/10', '60/20/20'],
    'Shock F1': [
        round(f1_score(y_test_lr, pred_lr2, average=None, labels=[2], zero_division=0)[0], 4),
        round(f1_score(y_test_8010, pred_8010, average=None, labels=[2], zero_division=0)[0], 4),
        round(f1_score(y_test_6020, pred_6020, average=None, labels=[2], zero_division=0)[0], 4),
    ],
    'Shock Recall': [
        round(recall_score(y_test_lr, pred_lr2, average=None, labels=[2], zero_division=0)[0], 4),
        round(recall_score(y_test_8010, pred_8010, average=None, labels=[2], zero_division=0)[0], 4),
        round(recall_score(y_test_6020, pred_6020, average=None, labels=[2], zero_division=0)[0], 4),
    ],
    'Overall Accuracy': [
        round(accuracy_score(y_test_lr, pred_lr2), 4),
        round(accuracy_score(y_test_8010, pred_8010), 4),
        round(accuracy_score(y_test_6020, pred_6020), 4),
    ]
})

print("SPLIT COMPARISON Logistic Regression (C=0.01)")
print()
print(split_summary_df.to_string(index=False))


In [ ]:
split_labels  = ['70/15/15', '80/10/10', '60/20/20']
shock_f1_vals = [
    round(f1_score(y_test_lr,   pred_lr2,   average=None, labels=[2], zero_division=0)[0], 4),
    round(f1_score(y_test_8010, pred_8010,  average=None, labels=[2], zero_division=0)[0], 4),
    round(f1_score(y_test_6020, pred_6020,  average=None, labels=[2], zero_division=0)[0], 4),
]
shock_rec_vals = [
    round(recall_score(y_test_lr,   pred_lr2,   average=None, labels=[2], zero_division=0)[0], 4),
    round(recall_score(y_test_8010, pred_8010,  average=None, labels=[2], zero_division=0)[0], 4),
    round(recall_score(y_test_6020, pred_6020,  average=None, labels=[2], zero_division=0)[0], 4),
]
high_f1_vals = [
    round(f1_score(y_test_lr,   pred_lr2,   average=None, labels=[0], zero_division=0)[0], 4),
    round(f1_score(y_test_8010, pred_8010,  average=None, labels=[0], zero_division=0)[0], 4),
    round(f1_score(y_test_6020, pred_6020,  average=None, labels=[0], zero_division=0)[0], 4),
]
acc_vals = [
    round(accuracy_score(y_test_lr,   pred_lr2),  4),
    round(accuracy_score(y_test_8010, pred_8010), 4),
    round(accuracy_score(y_test_6020, pred_6020), 4),
]

x_sp  = np.arange(3)
w_sp  = 0.20
colors = ['steelblue', 'darkorange', 'tomato', 'mediumseagreen']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

b1 = axes[0].bar(x_sp - w_sp,   shock_f1_vals,  w_sp, label='Shock F1',     color='tomato',      edgecolor='black')
b2 = axes[0].bar(x_sp,          shock_rec_vals, w_sp, label='Shock Recall',  color='lightsalmon', edgecolor='black')
b3 = axes[0].bar(x_sp + w_sp,   high_f1_vals,   w_sp, label='High F1',      color='darkorange',  edgecolor='black')

axes[0].set_xticks(x_sp)
axes[0].set_xticklabels(split_labels)
axes[0].set_ylabel('Score')
axes[0].set_title('Shock & High Class Metrics by Split')
axes[0].set_ylim(0, 1.0)
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

for bar in list(b1) + list(b2) + list(b3):
    h = bar.get_height()
    axes[0].annotate(f'{h:.3f}',
                     xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords='offset points',
                     ha='center', va='bottom', fontsize=8)

bar_colors = ['steelblue', 'darkorange', 'tomato']
b4 = axes[1].bar(split_labels, acc_vals, color=bar_colors, edgecolor='black', width=0.4)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Overall Accuracy by Split')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

for bar in b4:
    h = bar.get_height()
    axes[1].annotate(f'{h:.4f}',
                     xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords='offset points',
                     ha='center', va='bottom', fontsize=10)

plt.suptitle('Logistic Regression (C=0.01) — Split Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#OBSERVATIONS
1. Shock Detection (Primary Goal):
The 70/15/15 split is the original baseline. 80/10/10 gives the model more training data which can help it learn the minority Shock class better, but the 10% test set is too small to trust the resulting score. 60/20/20 balances both sides, reasonable training size and a test set large enough to produce stable Shock F1 scores.
2. Overall Accuracy:
70/15/15 and 60/20/20 both evaluate on enough samples to give a reliable accuracy figure. 80/10/10 evaluates on the fewest samples so its accuracy can swing from just one or two wrong predictions, making it the least trustworthy of the three.
3. Train Size vs Test Reliability Tradeoff:
70/15/15 is a well-established standard split and works reasonably here. 80/10/10 maximizes training data but at the cost of a dangerously small test set for a dataset of this size. 60/20/20 goes the other direction, it tests on the most samples, which matters more than the extra training rows when the overall dataset is small.
4. High Class Performance:
High F1 remains low across all three splits. This is not a splitting problem, it is a class imbalance problem. SMOTE is applied during training but the High class sits between Normal and Shock in terms of percentage change, making it genuinely difficult for the model to separate regardless of how the data is divided.

#RECOMMENDED SPLIT: 60/20/20


#Advancement
Linear Regression
Predicting Next Month Bill's Units

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

lr_features = ['units_previous', 'ac_hours', 'ac_count', 'fan_count',
               'fridge_age', 'light_hours', 'load_shedding_hours', 'people_count']

df_linreg = df[lr_features + ['units_latest']].dropna()

X_linreg = df_linreg[lr_features]
y_linreg = df_linreg['units_latest']


In [ ]:
X_train_linreg, X_test_linreg, y_train_linreg, y_test_linreg = train_test_split(
    X_linreg, y_linreg, test_size=0.2, random_state=42
)

linreg_model = LinearRegression()
linreg_model.fit(X_train_linreg, y_train_linreg)


In [ ]:
y_pred_linreg = linreg_model.predict(X_test_linreg)

print(f"R² Score : {r2_score(y_test_linreg, y_pred_linreg):.4f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_test_linreg, y_pred_linreg)):.2f}")

coef_df_linreg = pd.DataFrame({'Feature': lr_features, 'Coefficient': linreg_model.coef_})
print("\nCoefficients:\n", coef_df_linreg.sort_values('Coefficient', ascending=False).to_string(index=False))


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test_linreg, y_pred_linreg, alpha=0.5, edgecolors='k', linewidths=0.4)
plt.plot([y_test_linreg.min(), y_test_linreg.max()],
         [y_test_linreg.min(), y_test_linreg.max()], 'r--', lw=2)
plt.xlabel('Actual Units (units_latest)')
plt.ylabel('Predicted Units')
plt.title('Linear Regression: Actual vs Predicted Current Units')
plt.tight_layout()
plt.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("NEXT MONTH BILL PREDICTOR")

linreg_people_w       = widgets.IntSlider(value=4, min=1, max=20, description='People:')
linreg_ac_count_w     = widgets.IntSlider(value=1, min=0, max=10, description='AC Count:')
linreg_fan_count_w    = widgets.IntSlider(value=3, min=0, max=20, description='Fans:')

linreg_units_prev_w   = widgets.FloatText(value=250.0, description='Prev Units:')
linreg_units_latest_w = widgets.FloatText(value=300.0, description='Latest Units:')
linreg_ac_hours_w     = widgets.FloatText(value=6.0,   description='AC hrs/day:')
linreg_fridge_age_w   = widgets.FloatText(value=3.0,   description='Fridge Age:')
linreg_light_hours_w  = widgets.FloatText(value=6.0,   description='Light hrs/day:')
linreg_loadshed_w     = widgets.FloatText(value=2.0,   description='Load Shed hrs:')

linreg_predict_btn = widgets.Button(description='Predict Next Month', button_style='success')
linreg_output      = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<b>Household</b>"),
    linreg_people_w, linreg_ac_count_w, linreg_fan_count_w,
    widgets.HTML("<b>Recent Bills</b>"),
    linreg_units_prev_w, linreg_units_latest_w,
    widgets.HTML("<b>Appliance Usage</b>"),
    linreg_ac_hours_w, linreg_fridge_age_w, linreg_light_hours_w, linreg_loadshed_w,
    linreg_predict_btn, linreg_output
]))

def on_predict_linreg(b):
    with linreg_output:
        clear_output()

        trend              = linreg_units_latest_w.value - linreg_units_prev_w.value
        projected_previous = linreg_units_latest_w.value + trend

        input_df_linreg = pd.DataFrame([{
            'units_previous'     : projected_previous,
            'ac_hours'           : linreg_ac_hours_w.value,
            'ac_count'           : linreg_ac_count_w.value,
            'fan_count'          : linreg_fan_count_w.value,
            'fridge_age'         : linreg_fridge_age_w.value,
            'light_hours'        : linreg_light_hours_w.value,
            'load_shedding_hours': linreg_loadshed_w.value,
            'people_count'       : linreg_people_w.value,
        }])

        predicted = linreg_model.predict(input_df_linreg[lr_features])[0]
        delta     = predicted - linreg_units_latest_w.value
        direction = "Higher" if delta > 0 else "Lower"

        print("NEXT MONTH PREDICTION")
        print(f"  Previous Bill  : {linreg_units_prev_w.value:.0f} units")
        print(f"  Latest Bill    : {linreg_units_latest_w.value:.0f} units")
        print(f"  Trend          : {abs(trend):.0f} units/month")
        print(f"  Next month     : {predicted:.0f} units")
        print(f"  Change         : {direction} by {abs(delta):.0f} units")
        print()

        if predicted < 200:
            tier = "Low! well within normal range."
        elif predicted < 400:
            tier = "Medium! average household usage."
        else:
            tier = "High! consider reducing AC/light hours."

        print(f"  {tier}")

linreg_predict_btn.on_click(on_predict_linreg)
